# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write Rust quantized samples in `audio_samples/<model>.rs`.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
from typing import TYPE_CHECKING

from utils import (
    TARGET_AUDIO_LEN_TIME,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "cnn_time_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs
BATCH_SIZE = 32

In [32]:
from utils import make_time_datasets

train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)

Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']


In [33]:
model = keras.Sequential([
    # Enter as rank-2 tensor (batch, time), then expand to 4D for Conv2D.
    keras.layers.Input(shape=(TARGET_AUDIO_LEN_TIME,)),
    keras.layers.Reshape((TARGET_AUDIO_LEN_TIME, 1, 1), name="to_4d"),
    keras.layers.Conv2D(4, (3, 1), activation="relu"),
    keras.layers.AveragePooling2D(pool_size=(2, 1), strides=(2, 1)),
    keras.layers.Conv2D(8, (3, 1), activation="relu"),
    keras.layers.GlobalAveragePooling2D(keepdims=True, name="final_pool"),
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])


model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ to_4d (Reshape)                 │ (None, 47872, 1, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 47870, 1, 4)    │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_5             │ (None, 23935, 1, 4)    │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 23933, 1, 8)    │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_pool                      │ (None, 1, 1, 8)        │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 826 (3.23 KB)

 Trainable params: 826 (3.23 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
from utils import init_wandb, get_callbacks, finish_wandb

init_wandb(MODEL_STEM, config={
    "conv1_filters": 4,
    "conv2_filters": 8,
    "kernel_size": 3,
    "dense_units": 64,
})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=50,
    callbacks=get_callbacks(10,5, BATCH_SIZE)
)
finish_wandb()

Epoch 1/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.6689 - loss: 0.6281

/home/nathan/Documents/tiny-chirp-microflow/building_tensorflow/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


353/353 ━━━━━━━━━━━━━━━━━━━━ 23s 59ms/step - accuracy: 0.6773 - loss: 0.5818 - val_accuracy: 0.7362 - val_loss: 0.4746
Epoch 2/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - accuracy: 0.7668 - loss: 0.4508 - val_accuracy: 0.8674 - val_loss: 0.3987
Epoch 3/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 18s 52ms/step - accuracy: 0.8038 - loss: 0.3871 - val_accuracy: 0.9065 - val_loss: 0.3266
Epoch 4/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 52ms/step - accuracy: 0.8604 - loss: 0.3259 - val_accuracy: 0.9014 - val_loss: 0.2749
Epoch 5/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - accuracy: 0.8828 - loss: 0.2914 - val_accuracy: 0.9239 - val_loss: 0.2421
Epoch 6/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - accuracy: 0.8947 - loss: 0.2748 - val_accuracy: 0.9203 - val_loss: 0.2274
Epoch 7/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.9021 - loss: 0.2616 - val_accuracy: 0.9116 - val_loss: 0.2170
Epoch 8/100
353/353 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.9092 - loss: 0.2493 - val_

batch/accuracy,▁▁▃▄▆▇▇▇▇█▇▇▇█████████████████████████▇█
batch/batch_step,▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,██▃▃▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▂▅▆▇▇▇▇▇▇▇▇▇█▇█████████████████████████
epoch/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▇▇▇▇██████████▇███████████████████████▇
epoch/val_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▂▂▁▁▁▁▁▁▁▂▁▁▂
+6,...


In [35]:
from utils import (
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

export_inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME,), name="audio_waveform")
export_outputs = model(export_inputs)
export_model = keras.Model(export_inputs, export_outputs, name="cnn_time_microflow")

val_specs = build_representative_batches(test_ds, take=100)

export_keras_model_to_int8_tflite(export_model, val_specs, OUT_TFLITE)

Saved artifact at '/tmp/tmpwpa96qm2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 47872), dtype=tf.float32, name=None)
Output Type:
  TensorSpec(shape=(1, 2), dtype=tf.float32, name=None)
Captures:
  129835656474720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654872080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654870848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654864336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654872608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654872432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654871200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129835654871728: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1777031582.305622   71373 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1777031582.305640   71373 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [ ]:
from utils import evaluate_tflite_model, SAMPLE_RATE

hyperparams = {
    "conv1_filters": 4,
    "conv1_kernel": [3, 1],
    "conv2_filters": 8,
    "conv2_kernel": [3, 1],
    "pool_size": [2, 1],
    "dense_hidden": 64,
    "target_audio_len": TARGET_AUDIO_LEN_TIME,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")